# Random Forest

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm


In [20]:
def gini(y):
    if len(y) == 0:
        return 0.0
    p1 = np.mean(y)
    p0 = 1 - p1
    return 1 - (p0 ** 2 + p1 ** 2)

def best_split(X, y, max_thresholds=20):
    m, n = X.shape
    best_imp = np.inf
    best_feat = None
    best_thr = None
    for feat in range(n):
        vals = X[:, feat]
        unique_vals = np.unique(vals)
        if len(unique_vals) <= 1:
            continue
        if len(unique_vals) > max_thresholds:
            percentiles = np.linspace(0, 100, max_thresholds + 2)[1:-1]
            thresholds = np.unique(np.percentile(vals, percentiles))
        else:
            thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2.0
        for thr in thresholds:
            left_mask = vals <= thr
            n_l = np.count_nonzero(left_mask)
            n_r = m - n_l
            if n_l == 0 or n_r == 0:
                continue
            y_l = y[left_mask]
            y_r = y[~left_mask]
            impurity = (n_l * gini(y_l) + n_r * gini(y_r)) / m
            if impurity < best_imp:
                best_imp, best_feat, best_thr = impurity, feat, thr
    return best_feat, best_thr, best_imp

In [ ]:
class TreeNode:
    def __init__(self, depth=0, max_depth=None, min_samples_split=2):
        self.depth = depth
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.feature = None
        self.threshold = None
        self.left = None
        self.right = None
        self.pred_proba = None  
        self.feature_subset = None

    def fit(self, X, y):
        if (self.max_depth is not None and self.depth >= self.max_depth) or len(y) < self.min_samples_split or np.unique(y).size == 1:
            self.pred_proba = np.mean(y)
            return
        feat, thr, _ = best_split(X, y)
        if feat is None:
            self.pred_proba = np.mean(y)
            return
        self.feature = feat
        self.threshold = thr
        left_idx = X[:, feat] <= thr
        right_idx = ~left_idx
        self.left = TreeNode(depth=self.depth+1, max_depth=self.max_depth, min_samples_split=self.min_samples_split)
        self.right = TreeNode(depth=self.depth+1, max_depth=self.max_depth, min_samples_split=self.min_samples_split)
        self.left.fit(X[left_idx], y[left_idx])
        self.right.fit(X[right_idx], y[right_idx])

    def predict_proba_one(self, x):
        if self.pred_proba is not None:
            return self.pred_proba
        if x[self.feature] <= self.threshold:
            return self.left.predict_proba_one(x)
        else:
            return self.right.predict_proba_one(x)

    def predict_proba(self, X):
        return np.array([self.predict_proba_one(row) for row in X])

class RandomForest:
    def __init__(self, n_estimators=30, max_depth=10, min_samples_split=5, max_features='sqrt', random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        np.random.seed(random_state)

    def _feature_subset(self, n_features):
        if self.max_features == 'sqrt':
            k = max(1, int(np.sqrt(n_features)))
        elif self.max_features == 'log2':
            k = max(1, int(np.log2(n_features)))
        else:
            k = n_features
        return np.random.choice(n_features, k, replace=False)

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.trees = []
        for i in range(self.n_estimators):
            idx = np.random.choice(n_samples, n_samples, replace=True)
            X_boot, y_boot = X[idx], y[idx]
            feat_idx = self._feature_subset(n_features)
            X_sub = X_boot[:, feat_idx]
            tree = TreeNode(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            tree.feature_subset = feat_idx
            tree.fit(X_sub, y_boot)
            self.trees.append(tree)

    def predict_proba(self, X):
        probas = np.zeros((X.shape[0], len(self.trees)))
        for i, tree in enumerate(self.trees):
            X_sub = X[:, tree.feature_subset]
            probas[:, i] = tree.predict_proba(X_sub)
        return np.mean(probas, axis=1)

    def predict(self, X, threshold=0.2):  
        probs = self.predict_proba(X)
        return (probs >= threshold).astype(int)

In [ ]:
# Metric helpers 
def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))
def precision(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    return float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0
def recall(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    return float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
def f1(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return float(2 * p * r / (p + r)) if (p + r) > 0 else 0.0


In [ ]:
# Load test data
test_df = pd.read_csv('data/data_encoded_test.csv')
X_test = test_df.drop(columns=['is_fraud']).values
y_test = test_df['is_fraud'].values.astype(int)

df = pd.read_csv('data/train_smote_tomek.csv')
X_train = df.drop(columns=['is_fraud']).values
y_train = df['is_fraud'].values.astype(int)
rf = RandomForest(n_estimators=30, max_depth=1000, min_samples_split=5, max_features='sqrt')
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Final Metrics:")
print(f'accuracy: {accuracy(y_test, y_pred)}')
print(f'recall: {recall(y_test, y_pred)}')
print(f'f1 score: {f1(y_test, y_pred)}')
print(f'precision: {precision(y_test, y_pred)}')


Final Metrics:
accuracy: 0.9648
recall: 0.8333333333333334
f1 score: 0.14563106796116507
precision: 0.0797872340425532
